# Model Seçimi İçin Validation/Cross-Validation Yöntemleri

Bu dersin sonunda şunları yapabiliyor olmayı hedefliyoruz:

1. Daha önce çalıştığımız notebooklarda oluşturduğumuz DataFrame'i kullanabilmek
2. Modele karar vermek için veriyi train/validation/test olarak bölebilmek
3. Validation veri seti üzerinde cross-validation uygulayabilmek

## 1. Oluşturduğumuz DataFrame'i Çağırma

In [1]:
from joblib import dump, load
import numpy as np
import pandas as pd

cars = load('C:/Users/ataoz/Documents/GitHub/DSBootcampTürkçe/proje2/linear-regression-code-intro/data/cars_data.pkl')

In [2]:
cars

,normalized-losses,num-of-doors,wheel_base,length,width,height,curb_weight,num_of_cylinders,engine_size,bore,...,make[T.mitsubishi],make[T.nissan],make[T.peugot],make[T.plymouth],make[T.porsche],make[T.saab],make[T.subaru],make[T.toyota],make[T.volkswagen],make[T.volvo]
3,164.0,4,99.8,176.6,66.2,54.3,2337,4,109,3.19,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,164.0,4,99.4,176.6,66.4,54.3,2824,5,136,3.19,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,158.0,4,105.8,192.7,71.4,55.7,2844,5,136,3.19,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,158.0,4,105.8,192.7,71.4,55.9,3086,5,131,3.13,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10,192.0,2,101.2,176.8,64.8,54.3,2395,4,108,3.50,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200,95.0,4,109.1,188.8,68.9,55.5,2952,4,141,3.78,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
201,95.0,4,109.1,188.8,68.8,55.5,3049,4,141,3.78,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
202,95.0,4,109.1,188.8,68.9,55.5,3012,6,173,3.58,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
203,95.0,4,109.1,188.8,68.9,55.5,3217,6,145,3.01,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


Veri setimiz hazır olduğuna göre modelleme aşamasına geçebiliriz. Öğrendiklerimizden yola çıkarak, 3 model arasından seçim yapmak için validation sürecini uygulayalım. İşte kullanacağımız algoritmalar;
- Lineer Regresyon
- Ridge Regresyon
- 2.Derece Polinomal Regresyon

## 2. Basit Validation İşlemi: Train / Validation / Test

Burada veri setimizi 3 parçaya ayırıyoruz;
- Modelin eğitimi için **%60**
- Validation aşaması için **%20** (modeli seçmek için kullanılır)
- Son olarak test değerlendirmesi için **%20**

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler, PolynomialFeatures

X = cars.drop(['price','log_price','height','compression-ratio','normalized-losses','num-of-doors','stroke','peak-rpm'],1) 
y = cars['log_price']

# Veri setinin %20'lik kısmını Test aşaması için saklıyoruz
X, X_test, y, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [4]:
print("X:", X.shape)
print("y:", y.shape)
print("X_test:", X_test.shape)
print("y_test", y_test.shape)

X: (127, 28)
y: (127,)
X_test: (32, 28)
y_test (32,)


- Şimdi de **Validation** için kullanacağımız verileri bir kenara ayıralım.

In [5]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, random_state=42)

In [6]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

X_train: (95, 28)
y_train: (95,)
X_val: (32, 28)
y_val: (32,)


Şimdi bir model kurulumuna ihtiyacımız var:

- **Regularization** (ilerleyen derslerimizde ne olduğunu konuşacağız) kullanırken, tüm özelliklerin aynı ölçekte olması için verileri standartlaştırmalıyız (standartlaştırmayı da konuşacağız :)). Bu ölçekleme modelimizin bir parçası olduğu için, eğitim seti özellik dağılımlarını kullanarak ölçeklendirmemiz ve ölçekleyiciyi yeniden takmadan doğrulama ve test için aynı ölçeklendirmeyi uygulamamız gerekiyor. 

- Ayrıca, **polinomal model** oluşturmak için de polinom özellikleri almamız gerekiyor.

In [7]:
# Validation aşaması (model seçimi) için kullanacağımız 3 modeli oluşturma

# Lineer Regresyon
lm = LinearRegression()

# Ridge Regresyon
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train.values)
X_val_scaled = scaler.transform(X_val.values)
X_test_scaled = scaler.transform(X_test.values)

lm_reg = Ridge(alpha=1)

# Polinomal Regresyon
poly = PolynomialFeatures(degree=2) 

X_train_poly = poly.fit_transform(X_train.values)
X_val_poly = poly.transform(X_val.values)
X_test_poly = poly.transform(X_test.values)

lm_poly = LinearRegression()

- Bakalım nasıl sonuçlarla karşılaşacağız...

In [8]:
# Validation sonuçları

lm.fit(X_train, y_train)
print(f'Linear Regression val R^2: {lm.score(X_val, y_val):.3f}')

lm_reg.fit(X_train_scaled, y_train)
print(f'Ridge Regression val R^2: {lm_reg.score(X_val_scaled, y_val):.3f}')

lm_poly.fit(X_train_poly, y_train)
print(f'Degree 2 polynomial regression val R^2: {lm_poly.score(X_val_poly, y_val):.3f}')

Linear Regression val R^2: 0.780
Ridge Regression val R^2: 0.874
Degree 2 polynomial regression val R^2: -153.755


- Polinomal Regresyon modelimiz **Negatif R2** değeri döndürmüş, bu durum modelimizin **overfit** olduğunu gösterir! 

- Bu validation adımını gerçekleştirdikten sonra, **Ridge Regresyon** modelinin en iyi model olduğunu işaret ettiğini görüyoruz. Dolayısıyla validation aşamamız, **en başarılı modeli seçmemize** izin veriyor. 

- Son adım olarak da **Train&Validation** verilerini birleştirerek modelimizi tekrardan eğitelim ve **Test** veri setinde yakaladığımız başarıya bakalım.

In [9]:
# İşte modelimizin başarısı
X_scaled = scaler.fit_transform(X.values)
lm_reg.fit(X_scaled,y)
print(f'Ridge Regression test R^2: {lm_reg.score(X_test_scaled, y_test):.3f}')

Ridge Regression test R^2: 0.821


Gayet güzel! 

- Ancak **Cross-Validation** kullanarak bulduğumuz bu sonuçtan daha da emin olabiliriz.

## 3. Detaylı Validation İşlemi: Cross-Validation / Test

Burada verileri 2 parçaya ayıracağız: cross-validation süreci için %80 ve son test değerlendirmesi için %20.

Cross-Validation fikrinin, bize sunulan verileri verimli kullanmak (yukarıdaki %60 yerine %80 kullanarak) ve aynı zamanda **birden fazla validation kontrolü yapmak** için olduğunu unutmayın.

Basit olması için, bu aşamada **Lineer Regresyon** ve **Ridge Regresyonuna** odaklanacağız (Yukarıdaki kötü sonuçlara dayanarak tam derece 2. Dereceden Polinomal Regresyonunu atmak konusunda da oldukça rahat hissedebiliriz!) Cross-Validation parçalarımızda döngü yaparken, her iki modeli de eğitecek, doğrulacak ve sonunda karşılaştırmak için sonuçları toplayacağız.

In [10]:
from sklearn.model_selection import KFold

X, y = cars.drop('price',axis=1), cars['price']

X, X_test, y, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# Cross-validation index oluşturmasına yardımcı olmak için
X, y = np.array(X), np.array(y)

In [11]:
# Verimizi 5 parçaya ayırmaya deneyelim (n_splits=5)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Validation sonuçlarını toplayacağımız listeler
cv_lm_r2 = []
cv_lm_reg_r2 = []

for train_ind, val_ind in kf.split(X,y):
    
    X_train, y_train = X[train_ind], y[train_ind]
    X_val, y_val = X[val_ind], y[val_ind] 
    
    # Modellerin oluşturulması
    lm = LinearRegression()
    lm_reg = Ridge(alpha=1)

    # Modellerin eğitimi
    lm.fit(X_train, y_train)
    cv_lm_r2.append(lm.score(X_val, y_val))
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    
    lm_reg.fit(X_train_scaled, y_train)
    cv_lm_reg_r2.append(lm_reg.score(X_val_scaled, y_val))

# Sonuçların toplanması
print('Simple regression scores: ', cv_lm_r2)
print('Ridge scores: ', cv_lm_reg_r2, '\n')

print(f'Simple mean cv r^2: {np.mean(cv_lm_r2):.3f} +- {np.std(cv_lm_r2):.3f}')
print(f'Ridge mean cv r^2: {np.mean(cv_lm_reg_r2):.3f} +- {np.std(cv_lm_reg_r2):.3f}')

Simple regression scores:  [0.9467877648365283, 0.9564718915634055, 0.9406418211170544, 0.9552566123847734, 0.9440575290736788]
Ridge scores:  [0.9822968722280486, 0.9483622376662204, 0.9762223275761345, 0.9664196012874114, 0.94300253261] 

Simple mean cv r^2: 0.949 +- 0.006
Ridge mean cv r^2: 0.963 +- 0.015


- Cross-Validation'dan elde ettiğimiz sonuçları incelediğimizde aslında Lineer Regresyon ile Ridge Regresyon arasında modelin eğitiminde gördüğümüz kadar büyük bir fark olmadığını görüyoruz. Hatta, Lineer Regresyonda bulunan validation skorları Ridge'e göre daha bile tutarlı duruyor. 
- İşte bu durum bize model oluşturma aşamasında Cross-Validation işleminin önemini gösteriyor.